# Symbolic music sequence modeling
RNN/LSTM coursework experiments, inspired by the MIT Introduction to Deep Learning music-generation lab. Corpus-derived data is generated locally; no model weights or audio quality claim is bundled.


### Experiment cell 1


In [ ]:
from sys import dont_write_bytecode
# ============================================================
# MUSIC GENERATION WITH RNNs & LSTMs (COLAB VERSION)
# Inspired by MIT Deep Learning Lab 1: https://github.com/MITDeepLearning/introtodeeplearning/tree/master/lab1
# ============================================================
# Coursework-derived experiment; see PROVENANCE.md for attribution.

# SECTION 0: Setup
# ============================================================


import gc
import random
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import pretty_midi
import glob
import os
import matplotlib.pyplot as plt
from scipy.io.wavfile import write as wav_write
from IPython.display import display, Audio

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Keep notebook training stable on macOS/Jupyter by avoiding aggressive backends.
try:
    torch.set_num_threads(1)
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

if hasattr(torch.backends, "mkldnn"):
    torch.backends.mkldnn.enabled = False

print("PyTorch version:", torch.__version__)
device = torch.device("cpu")
print("Using device:", device, "(forced to CPU for notebook stability)")

### Experiment cell 2


In [ ]:
# ============================================================
# 🎓 SECTION 1: What are we doing?
# ============================================================

"""
🎯 GOAL:
Train a neural network to generate music by predicting the next note in a sequence.

We treat music like language:
- Notes = words
- Song = sentence
- Model learns patterns over time

We will:
1. Load MIDI files
2. Convert them into sequences of notes
3. Train an RNN/LSTM
4. Generate new music 🎶
"""

### Experiment cell 3


In [ ]:
# ============================================================
# SECTION 2: Generate a small MIDI dataset (Robust Method)
# ============================================================
import music21

os.makedirs("midi_dataset", exist_ok=True)

MAX_CHORALES = 250
print("Loading Bach chorales from music21 corpus...")

chorale_scores = []
midi_files = []

for i, score in enumerate(list(music21.corpus.chorales.Iterator())[:MAX_CHORALES]):
    midi_path = f"midi_dataset/bach_{i:03d}.mid"
    try:
        score.write('midi', fp=midi_path)
        chorale_scores.append(score)
        midi_files.append(midi_path)
    except Exception:
        continue

print("Number of Bach chorales loaded:", len(chorale_scores))
print("Number of MIDI files written:", len(midi_files))

### Experiment cell 4


In [ ]:
# ============================================================
# SECTION 3: MIDI -> Note Sequences & Visualization
# ============================================================

def normalize_score(score):
    try:
        detected_key = score.analyze('key')
        target_tonic = music21.pitch.Pitch('C') if detected_key.mode == 'major' else music21.pitch.Pitch('A')
        interval_to_target = music21.interval.Interval(detected_key.tonic, target_tonic)
        return score.transpose(interval_to_target)
    except Exception:
        return score

def event_to_token(event):
    duration = round(float(event.duration.quarterLength), 2)
    if duration <= 0:
        return None
    if event.isRest:
        return f"R|{duration}"
    if event.isChord:
        highest_pitch = max(pitch.midi for pitch in event.pitches)
        return f"{highest_pitch}|{duration}"
    return f"{event.pitch.midi}|{duration}"

def score_to_tokens(score):
    normalized_score = normalize_score(score)
    melody_part = normalized_score.parts[0] if normalized_score.parts else normalized_score
    tokens = []

    for event in melody_part.flatten().notesAndRests:
        token = event_to_token(event)
        if token is not None:
            tokens.append(token)

    return tokens

def token_to_pitch(token):
    pitch_text, _ = token.split("|")
    return np.nan if pitch_text == "R" else int(pitch_text)

all_token_sequences = []
for score in chorale_scores:
    tokens = score_to_tokens(score)
    if len(tokens) > 24:
        all_token_sequences.append(tokens)

all_tokens = [token for sequence in all_token_sequences for token in sequence]

print("Number of usable chorales:", len(all_token_sequences))
print("Total tokens collected:", len(all_tokens))
print("Unique tokens:", len(set(all_tokens)))

preview_pitches = [token_to_pitch(token) for token in all_tokens[:150]]

plt.figure(figsize=(12, 4))
plt.plot(preview_pitches, marker='o', linestyle='-', color='b', alpha=0.7, markersize=4)
plt.title("First 150 Melody Events (pitch over time, rests omitted in the line)")
plt.xlabel("Time Step")
plt.ylabel("MIDI Pitch")
plt.grid(True)
plt.show()

### Experiment cell 5


In [ ]:
# ============================================================
# SECTION 4: Create sequences (like sentences)
# ============================================================
SEED = 10
SEQ_LENGTH = 24

shuffled_sequences = list(all_token_sequences)
random.Random(SEED).shuffle(shuffled_sequences)

split_index = int(0.85 * len(shuffled_sequences))
train_sequences = shuffled_sequences[:split_index]
test_sequences = shuffled_sequences[split_index:]

token_counts = Counter(token for sequence in train_sequences for token in sequence)
idx_to_token = ["<PAD>", "<UNK>"] + sorted(token_counts)
token_to_idx = {token: idx for idx, token in enumerate(idx_to_token)}
vocab_size = len(idx_to_token)

def encode_sequence(tokens):
    return [token_to_idx.get(token, token_to_idx["<UNK>"]) for token in tokens]

def create_sequences(sequence_list, seq_length):
    X = []
    y = []

    for tokens in sequence_list:
        encoded = encode_sequence(tokens)
        for i in range(len(encoded) - seq_length):
            X.append(encoded[i:i + seq_length])
            y.append(encoded[i + seq_length])

    return np.array(X, dtype=np.int64), np.array(y, dtype=np.int64)

X_train, y_train = create_sequences(train_sequences, SEQ_LENGTH)
X_test, y_test = create_sequences(test_sequences, SEQ_LENGTH)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
print("Vocabulary size:", vocab_size)

### Experiment cell 6


In [ ]:
# Encode tokens as integer classes for categorical training
BATCH_SIZE = 64

X_train_tensor = torch.tensor(X_train, dtype=torch.long)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Training batches:", len(train_loader))
print("Test batches:", len(test_loader))

### Experiment cell 7


In [ ]:
# ============================================================
# SECTION 5: Build RNN model (PyTorch)
# ============================================================

import torch
import torch.nn as nn

# Define device for model training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EMBEDDING_DIM = 64
HIDDEN_SIZE = 128
NUM_LAYERS = 1
DROPOUT = 0.0
vocab_size = 128  # Assuming a standard MIDI vocabulary size, adjust as needed

class MusicRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=EMBEDDING_DIM, hidden_size=HIDDEN_SIZE):
        super(MusicRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=NUM_LAYERS,
            nonlinearity='tanh',
            dropout=DROPOUT,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        out, _ = self.rnn(embedded)
        logits = self.fc(out[:, -1, :])
        return logits

rnn_model = MusicRNN(vocab_size).to(device)
print(rnn_model)

### Experiment cell 8


In [ ]:
# ============================================================
# SECTION 6: LSTM (FULLY FIXED) & Training
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
import gc
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 🔥 CRITICAL FIX: vocab_size must match max index
# ------------------------------------------------------------
vocab_size = max(token_to_idx.values()) + 1

# Optional safety checks (run once)
print("Vocab size:", vocab_size)
print("Max idx in vocab:", max(token_to_idx.values()))

assert "<PAD>" in token_to_idx, "Missing <PAD> token"
assert "<UNK>" in token_to_idx, "Missing <UNK> token"


# ------------------------------------------------------------
# Model
# ------------------------------------------------------------
class MusicLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=EMBEDDING_DIM, hidden_size=HIDDEN_SIZE):
        super(MusicLSTM, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=NUM_LAYERS,
            dropout=DROPOUT if NUM_LAYERS > 1 else 0,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        # 🔒 Safety check (can remove later)
        if x.max() >= self.embedding.num_embeddings or x.min() < 0:
            raise ValueError(f"Invalid index in input: min={x.min()}, max={x.max()}")

        embedded = self.embedding(x)
        out, _ = self.lstm(embedded)
        logits = self.fc(out[:, -1, :])
        return logits


# ------------------------------------------------------------
# Loss
# ------------------------------------------------------------
criterion = nn.CrossEntropyLoss()


# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------
def evaluate_model(model, data_loader):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for batch_X, batch_y in data_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            logits = model(batch_X)
            loss = criterion(logits, batch_y)

            total_loss += loss.item() * batch_X.size(0)

    return total_loss / len(data_loader.dataset)


# ------------------------------------------------------------
# Training
# ------------------------------------------------------------
def train_model(model, train_loader, test_loader, epochs=20, lr=0.001):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = {"train": [], "test": []}

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0.0

        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            # 🔒 Clamp to prevent crashes (safety)
            batch_X = torch.clamp(batch_X, 0, vocab_size - 1)
            batch_y = torch.clamp(batch_y, 0, vocab_size - 1)

            optimizer.zero_grad(set_to_none=True)

            logits = model(batch_X)
            loss = criterion(logits, batch_y)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_train_loss += loss.item() * batch_X.size(0)

        avg_train_loss = total_train_loss / len(train_loader.dataset)
        avg_test_loss = evaluate_model(model, test_loader)

        history["train"].append(avg_train_loss)
        history["test"].append(avg_test_loss)

        print(f"Epoch {epoch+1}/{epochs} | Train: {avg_train_loss:.4f} | Test: {avg_test_loss:.4f}")
        gc.collect()

    return history


# ------------------------------------------------------------
# Sampling (FIXED: prevents repetition)
# ------------------------------------------------------------
def sample_next_token(logits, temperature=1.1, top_k=20,
                      recent_tokens=None, repetition_penalty=1.2):

    logits = logits.clone()

    # 🔥 Repetition penalty
    if recent_tokens is not None:
        for token in set(recent_tokens[-20:]):
            logits[token] = torch.where(logits[token] < 0, logits[token] * repetition_penalty, logits[token] / repetition_penalty)

    # Temperature
    logits = logits / max(temperature, 1e-6)

    # Top-k
    top_k = min(top_k, logits.shape[-1])
    top_logits, top_indices = torch.topk(logits, k=top_k)

    probs = torch.softmax(top_logits, dim=-1)

    idx = torch.multinomial(probs, 1).item()
    return top_indices[idx].item()


# ------------------------------------------------------------
# Generation (FIXED)
# ------------------------------------------------------------
def generate_notes(model, seed_tokens, n_notes=80,
                   temperature=1.1, top_k=20):

    model.eval()

    generated_tokens = list(seed_tokens)
    generated_indices = [
        token_to_idx.get(t, token_to_idx["<UNK>"]) for t in seed_tokens
    ]

    with torch.no_grad():
        for _ in range(n_notes):

            input_indices = generated_indices[-SEQ_LENGTH:]

            # 🔥 Padding FIX
            if len(input_indices) < SEQ_LENGTH:
                pad = [token_to_idx["<PAD>"]] * (SEQ_LENGTH - len(input_indices))
                input_indices = pad + input_indices

            input_tensor = torch.tensor([input_indices], dtype=torch.long).to(device)

            logits = model(input_tensor)[0].detach().cpu()

            next_idx = sample_next_token(
                logits,
                temperature=temperature,
                top_k=top_k,
                recent_tokens=generated_indices
            )

            generated_indices.append(next_idx)

            if next_idx in idx_to_token:
                generated_tokens.append(idx_to_token[next_idx])
            else:
                generated_tokens.append("<UNK>")

    return generated_tokens


# ------------------------------------------------------------
# 🔥 Train model
# ------------------------------------------------------------
EPOCHS = 20

lstm_model = MusicLSTM(vocab_size).to(device)

lstm_history = train_model(
    lstm_model,
    train_loader,
    test_loader,
    epochs=EPOCHS,
    lr=0.001
)


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
plt.figure(figsize=(8, 4))
plt.plot(range(1, EPOCHS + 1), lstm_history["train"], marker='o', label='Train')
plt.plot(range(1, EPOCHS + 1), lstm_history["test"], marker='s', label='Test')
plt.title("LSTM Train/Test Loss")
plt.xlabel("Epoch")
plt.ylabel("Cross Entropy Loss")
plt.legend()
plt.grid(True)
plt.show()

### Experiment cell 9


In [ ]:
# ============================================================
# SECTION: RNN vs LSTM Comparison
# ============================================================
print("Training RNN for comparison...")
rnn_model = MusicRNN(vocab_size).to(device)
rnn_history = train_model(rnn_model, train_loader, test_loader, epochs=EPOCHS, lr=0.001)

plt.figure(figsize=(10, 5))
plt.plot(range(1, EPOCHS + 1), rnn_history["train"],  marker='o',                label='RNN Train',  color='blue')
plt.plot(range(1, EPOCHS + 1), rnn_history["test"],   marker='o', linestyle='--', label='RNN Test',   color='skyblue')
plt.plot(range(1, EPOCHS + 1), lstm_history["train"], marker='s',                label='LSTM Train', color='red')
plt.plot(range(1, EPOCHS + 1), lstm_history["test"],  marker='s', linestyle='--', label='LSTM Test',  color='salmon')
plt.title("Training Loss Comparison: RNN vs LSTM")
plt.xlabel("Epoch")
plt.ylabel("Cross Entropy Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
comparison_plot_path = "rnn_lstm_train_test_loss.png"
plt.savefig(comparison_plot_path, dpi=200)
plt.show()
print(f"Saved comparison plot to {comparison_plot_path}")

# ------------------------------------------------------------
# Seed & generation
# generate_notes returns seed + new tokens, so:
#   full_sequence[:seed_len] = seed (for plot continuity)
#   full_sequence[seed_len:] = newly generated notes
# ------------------------------------------------------------
seed_tokens    = test_sequences[0][:SEQ_LENGTH] if len(test_sequences) > 0 else train_sequences[0][:SEQ_LENGTH]
rnn_generated  = generate_notes(rnn_model,  seed_tokens, n_notes=80, temperature=0.9, top_k=8)
lstm_generated = generate_notes(lstm_model, seed_tokens, n_notes=80, temperature=0.9, top_k=8)


# ------------------------------------------------------------
# Safe token → pitch converter
# Skips special tokens (<UNK>, <PAD>) that have no "|"
# ------------------------------------------------------------
def token_sequence_to_pitch_list(tokens):
    pitches = []
    for token in tokens:
        parts = token.split("|")
        if len(parts) < 2:
            pitches.append(np.nan)
            continue
        pitch_text = parts[0]
        pitches.append(np.nan if pitch_text == "R" else int(pitch_text))
    return pitches


seed_len     = len(seed_tokens)

seed_pitches = token_sequence_to_pitch_list(seed_tokens)
rnn_pitches  = token_sequence_to_pitch_list(rnn_generated)
lstm_pitches = token_sequence_to_pitch_list(lstm_generated)

rnn_new      = rnn_pitches[seed_len:]
lstm_new     = lstm_pitches[seed_len:]

rnn_x        = range(seed_len, seed_len + len(rnn_new))
lstm_x       = range(seed_len, seed_len + len(lstm_new))

plt.figure(figsize=(12, 4))
plt.plot(range(seed_len), seed_pitches,
         label="Seed",           color='black', linewidth=2)
plt.plot(rnn_x,  rnn_new,
         label="RNN Generated",  color='blue',  alpha=0.8)
plt.plot(lstm_x, lstm_new,
         label="LSTM Generated", color='red',   alpha=0.8)
plt.title("Generated Melody Comparison")
plt.xlabel("Time Step")
plt.ylabel("MIDI Pitch")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

### Experiment cell 10


In [ ]:
# ============================================================
# 1. Temperature-Controlled Generation (Categorical Output)
# ============================================================
import torch.nn.functional as F

temperature_trials = [0.7, 0.9, 1.1]
trial_scores = {}

for temperature in temperature_trials:
    temp_sample = generate_notes(lstm_model, seed_tokens, n_notes=32, temperature=temperature, top_k=8)
    continuation = temp_sample[SEQ_LENGTH:]
    unique_ratio = len(set(continuation)) / len(continuation)
    trial_scores[temperature] = unique_ratio
    print(f"Temperature {temperature:.1f} -> unique continuation ratio: {unique_ratio:.3f}")

BEST_TEMPERATURE = 0.9
BEST_TOP_K = 8
print(f"Chosen sampling setup: temperature={BEST_TEMPERATURE}, top_k={BEST_TOP_K}")

### Experiment cell 11


In [ ]:
# ============================================================
# 🥁 2. Multi-Instrument Modeling (Piano Roll)
# ============================================================
# Real music isn't just one note! We need a 2D matrix (Time x Pitch)

def get_multi_instrument_piano_roll(midi_file_path, fs=10):
    """
    Converts a MIDI file to a multi-track piano roll matrix.
    fs = frames per second (time resolution)
    """
    midi_data = pretty_midi.PrettyMIDI(midi_file_path)

    # Extract piano roll for all instruments (Shape: 128 pitches x Time)
    piano_roll = midi_data.get_piano_roll(fs=fs)

    # Binarize it: 1 if ANY instrument is playing that note, 0 otherwise
    piano_roll = (piano_roll > 0).astype(np.float32)

    # Transpose to (Time, 128) for the neural network input
    return piano_roll.T

if len(midi_files) > 0:
    sample_roll = get_multi_instrument_piano_roll(midi_files[0])
    print(f"🎹 Piano Roll Shape (Time Steps, 128 Pitches): {sample_roll.shape}")

    # Plot a snippet of the piano roll (e.g., first 100 time steps)
    plt.figure(figsize=(12, 4))
    plt.imshow(sample_roll[:100, :].T, aspect='auto', origin='lower', cmap='magma')
    plt.title("Piano Roll Representation (Multiple notes at once!)")
    plt.ylabel("MIDI Pitch")
    plt.xlabel("Time Step")
    plt.show()

### Experiment cell 12


In [ ]:
# ============================================================
# 3. Spectrogram-Based Audio Processing
# ============================================================
try:
    import torchaudio

    def create_spectrogram(wav_file_path):
        # Loads raw audio and converts it to a Mel Spectrogram.
        # This allows a CNN/Transformer to learn the actual sound (timbre, tone),
        # not just the symbolic notes.
        waveform, sample_rate = torchaudio.load(wav_file_path)

        mel_spectrogram_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=1024,
            hop_length=512,
            n_mels=64
        )

        mel_spec = mel_spectrogram_transform(waveform)
        mel_spec_db = torchaudio.transforms.AmplitudeToDB()(mel_spec)
        return mel_spec_db

    if os.path.exists('best_bach_music.wav'):
        spec = create_spectrogram('best_bach_music.wav')
        print(f"Spectrogram Shape: {spec.shape}")

        plt.figure(figsize=(12, 4))
        plt.imshow(spec[0].numpy(), aspect='auto', origin='lower', cmap='viridis')
        plt.title("Mel Spectrogram of Generated Audio")
        plt.ylabel("Mel Frequency Bins")
        plt.xlabel("Time Frames")
        plt.colorbar(format='%+2.0f dB')
        plt.show()
    else:
        print("Generate the final .wav file first to see the spectrogram!")
except Exception as exc:
    print(f"Skipping spectrogram cell because torchaudio is unavailable: {exc}")

### Experiment cell 13


In [ ]:
# ============================================================
# SECTION 7: Generate music (PyTorch)
# ============================================================

generated_tokens = generate_notes(
    lstm_model,
    seed_tokens,
    n_notes=120,
    temperature=BEST_TEMPERATURE,
    top_k=BEST_TOP_K
)

print("Generated sequence length:", len(generated_tokens))

plt.figure(figsize=(10, 4))
plt.plot(range(len(seed_tokens)), token_sequence_to_pitch_list(seed_tokens), label="Seed (Original)", color='blue')
plt.plot(
    range(len(seed_tokens), len(generated_tokens)),
    token_sequence_to_pitch_list(generated_tokens)[len(seed_tokens):],
    label="Generated by LSTM",
    color='orange'
)
plt.title("Seed vs. Generated Sequence")
plt.xlabel("Time Step")
plt.ylabel("MIDI Pitch")
plt.legend()
plt.grid(True)
plt.show()

### Experiment cell 14


In [ ]:
# ============================================================
# SECTION 8: Convert to MIDI
# ============================================================
def tokens_to_midi(tokens, filename="best_bach_music.mid"):
    midi = pretty_midi.PrettyMIDI()
    instrument = pretty_midi.Instrument(program=0)
    time = 0.0

    for token in tokens:
        # Skip special tokens that don't follow "pitch|duration" format
        parts = token.split("|")
        if len(parts) < 2:
            continue

        pitch_text, duration_text = parts[0], parts[1]

        try:
            duration = float(duration_text)
        except ValueError:
            continue  # Skip malformed duration

        if duration <= 0:
            continue  # Skip zero/negative durations

        if pitch_text != "R":
            try:
                pitch = max(0, min(127, int(pitch_text)))
            except ValueError:
                continue  # Skip malformed pitch
            note_obj = pretty_midi.Note(
                velocity=100,
                pitch=pitch,
                start=time,
                end=time + duration
            )
            instrument.notes.append(note_obj)

        time += duration

    midi.instruments.append(instrument)
    midi.write(filename)
    return filename


# Use lstm_generated as fallback if generated_tokens is not defined
if 'generated_tokens' not in dir():
    generated_tokens = lstm_generated  # or rnn_generated

best_midi_path = tokens_to_midi(generated_tokens, "best_bach_music.mid")
print(f"Saved as {best_midi_path}")

### Experiment cell 15


In [ ]:
# ============================================================
# SECTION 9: Play audio
# ============================================================

best_wav_path = "best_bach_music.wav"

pm = pretty_midi.PrettyMIDI(best_midi_path)
audio = pm.synthesize(fs=16000)
audio = audio / max(np.abs(audio).max(), 1e-8)
audio_int16 = np.asarray(audio * 32767, dtype=np.int16)
wav_write(best_wav_path, 16000, audio_int16)

print(f"Saved as {best_wav_path}")
display(Audio(best_wav_path))